In [5]:
import numpy as np
import random

class ShortDroneACO:
    def __init__(self, D, E, B_max, M=15, I=50, alpha=1.0, beta=2.0, rho=0.5, Q=100):
        self.D, self.E, self.B_max = np.array(D), np.array(E), B_max
        self.N, self.M, self.I = len(D), M, I
        self.alpha, self.beta, self.rho, self.Q = alpha, beta, rho, Q
        self.tau = np.ones((self.N, self.N)) * 0.1

    def solve(self):
        best_route, best_dist = None, float('inf')

        for _ in range(self.I):
            routes, distances = [], []
            for ant in range(self.M):
                curr, battery, dist, route = 0, self.B_max, 0, []
                unvisited = set(range(1, self.N))

                while unvisited:
                    safe = [n for n in unvisited if battery >= (self.E[curr][n] + self.E[n][0])]

                    if not safe:
                        dist += self.D[curr][0]
                        route.append(0)
                        curr, battery = 0, self.B_max
                    else:
                        probs = [(self.tau[curr][n]**self.alpha) * ((1.0/self.D[curr][n])**self.beta) for n in safe]
                        total = sum(probs)
                        next_node = random.choices(safe, weights=[p/total for p in probs], k=1)[0]

                        dist += self.D[curr][next_node]
                        battery -= self.E[curr][next_node]
                        route.append(next_node)
                        unvisited.remove(next_node)
                        curr = next_node

                dist += self.D[curr][0]
                route.append(0)
                routes.append(route)
                distances.append(dist)

                if dist < best_dist:
                    best_dist, best_route = dist, route

            self.tau *= (1.0 - self.rho)
            for r, d in zip(routes, distances):
                for i in range(len(r) - 1):
                    self.tau[r[i]][r[i+1]] += self.Q / d

        return best_route, best_dist

if __name__ == "__main__":
    print("--- Agricultural Drone Path Optimization Setup ---")

    # 1. User Input for Matrix Dimensions
    num_nodes = int(input("Enter total number of locations (Charging Station + Farm Regions): "))

    # 2. User Input for Distance Matrix
    print(f"\nEnter the Distance Matrix row by row (space-separated values).")
    print("Note: Node 0 is the Charging Station.")
    D_matrix = []
    for i in range(num_nodes):
        row = list(map(float, input(f"Row {i} elements ({num_nodes} values): ").split()))
        D_matrix.append(row)

    # 3. User Input for Energy Conversion Factor
    energy_factor = float(input("\nEnter energy consumption factor (e.g., 1.5 units of energy per 1 distance unit): "))
    E_matrix = [[val * energy_factor for val in row] for row in D_matrix]

    # 4. User Input for Battery Limit
    max_battery = float(input("Enter maximum drone battery capacity: "))

    # Run Algorithm
    optimizer = ShortDroneACO(D=D_matrix, E=E_matrix, B_max=max_battery)
    route, distance = optimizer.solve()

    print("\n--- Optimization Results ---")
    print(f"Optimal Route Sequence: {route}")
    print(f"Total Flight Distance: {distance} units")


--- Agricultural Drone Path Optimization Setup ---
Enter total number of locations (Charging Station + Farm Regions): 3

Enter the Distance Matrix row by row (space-separated values).
Note: Node 0 is the Charging Station.
Row 0 elements (3 values): 0 10 15
Row 1 elements (3 values):   10 0 20
Row 2 elements (3 values): 15 20 0

Enter energy consumption factor (e.g., 1.5 units of energy per 1 distance unit): 1.2
Enter maximum drone battery capacity: 40

--- Optimization Results ---
Optimal Route Sequence: [1, 0, 2, 0]
Total Flight Distance: 50.0 units
